In [26]:
import numpy as np
import matplotlib.pyplot as plt
import math as math

In [ ]:
atoms = 2000
Temp = 15000
kb = 1.38e-23
NA = 6.022e23
MSI = 28.0855/NA #use cgs system
angle = 0.3 #radians
sinangle = math.sin(angle)
cosangle = math.cos(angle)
thermsteps = 801
velarr = np.array([1e1,1e2,1e3,1e4,1e5,1e6,1e7,1e8])
casearr = np.array([0,1,2,3])
folder = "run10_8"

def trajtoarr(filename, dt,atoms=2000):
    print("run")
    timeflag =False
    atomflag=False
    count = 0
    atomlist = []
    tempatomlist=[]
    timestep=0
    with open(filename, 'r') as file:
        for line in file:
            if "ITEM" in line:
                if "ITEM: TIMESTEP" in line:
                    count += 1
                    timeflag = True
                    atomflag=False
                    tempatomlist.append(float(dt*timestep))
                    atomlist.append(tempatomlist)
                    tempatomlist=[]
                elif "ITEM: ATOMS id type vx vy vz" in line:
                    atomflag = True
                    timeflag=False
                else:
                    timeflag = False
                    atomflag = False
            if timeflag and ("ITEM" not in line):
                timestep = float(line.strip())
            if atomflag and ("ITEM" not in line):
                vz = float(line.strip().split(" ")[-1])
                vy = float(line.strip().split(" ")[-2])
                tempatomlist.append(sinangle*vy+cosangle*vz)
                count += 1
    tempatomlist.append(float(dt*timestep))
    atomlist.append(tempatomlist)
    atomlist.pop()
    atomlist.pop(0)
    atomlist.pop()
    atomarr = np.array(atomlist)
    print(atomlist)
    return atomarr
def logtoarr(filename):
    startlogstr = "   Step          Time           Temp         c_tempH        c_tempSi        TotEng         PotEng         KinEng         Press           v_k0           v_dt        v_collfreq       v_ccs "
    endlogstr = "Loop time of "
    templist=[]
    dataflag=False
    with open(filename, 'r') as file:
        for line in file:
            if dataflag:
                dataline = list(filter(None, line.split(" ")))
                dataline[-1] = dataline[-1][:-1]
                dataline = list(filter(None, dataline))
                templist.append(dataline)
            if startlogstr in line:
                dataflag=True
                templist=[]
            if endlogstr in line:
                dataflag=False

    templist.pop()
    templist.pop()
    templist.pop()
    templist.pop(0)    # print(templist)
    return np.array(templist,dtype=np.float64)
for i in range(len(velarr)):
    for j in range(len(casearr)):
        vel = velarr[i]
        case = casearr[j]
        datai = logtoarr(folder+"/unforcedslumlogvel_v{:.1e}_c{}.txt".format(vel,case))
        dt = (datai[1,1]-datai[0,1])/100
        print(dt)
        print(datai)
        trajarr = trajtoarr(folder+"/trajvel_v{:.1e}_c{}.txt".format(vel,case),dt,atoms=atoms)
        np.savetxt("log_v{:.1}_c{}.np".format(vel,case),datai)
        np.savetxt("force_v{:.1}_c{}.np".format(vel,case),trajarr)





8.474239399999999e-17
[[1.0000000e+02 8.4742396e-15 7.3823233e+03 ... 8.4742396e-17
  1.0495478e+15 1.3281700e-15]
 [2.0000000e+02 1.6948479e-14 6.9073150e+03 ... 8.4742396e-17
  1.0495478e+15 1.3281700e-15]
 [3.0000000e+02 2.5422719e-14 6.4475956e+03 ... 8.4742396e-17
  1.0495478e+15 1.3281700e-15]
 ...
 [7.6000000e+03 6.4404221e-13 4.9966210e+03 ... 8.4742396e-17
  1.0495478e+15 1.3281700e-15]
 [7.7000000e+03 6.5251645e-13 5.0007209e+03 ... 8.4742396e-17
  1.0495478e+15 1.3281700e-15]
 [7.8000000e+03 6.6099069e-13 5.0024485e+03 ... 8.4742396e-17
  1.0495478e+15 1.3281700e-15]]
run
[[-27846.961258318144, 40776.33452397473, -110089.14086323592, -114596.16752784862, -110506.55102313412, 240808.24380630677, 2354.7756042224355, -78932.8459140022, -29912.797317513658, 137896.41302874195, -63653.82100410952, -95975.75571047975, 61264.71839074924, -34650.25785962006, 190908.5653487879, 18442.07718011485, 2615.8123986371356, -178425.5313991985, -38316.2456754654, 94650.79411603382, 243639.356

# Plot Temp and Energy over time